**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Mechanistic Interpretability

Open the model and find the *mechanism*: we train a tiny transformer on a task with a known algorithm (detecting balanced parentheses), then locate where the network computes what — attention maps, linear probes, and the causal test that separates correlation from mechanism: **activation patching**.

## 1. Pre-requisites

[Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb), [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb); [Causal Inference](./Causal_Inference.ipynb) supplies the intervention mindset.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# task with a KNOWN algorithm: is a ()-string balanced? ground truth = running-depth check
VOCAB = {"(": 0, ")": 1, "PAD": 2}
L_seq = 16
def make_batch(B):
    xs, ys, depths = [], [], []
    for _ in range(B):
        if rng.random() < 0.5:                                # balanced: random matched string
            s = []
            depth = 0
            for i in range(L_seq):
                if depth == 0 or (rng.random() < 0.5 and depth < L_seq - i - depth):
                    s.append("("); depth += 1
                else:
                    s.append(")"); depth -= 1
            if depth > 0: s[-depth:] = [")"]*depth
        else:                                                  # corrupt one position
            s = ["(", ")"]*(L_seq//2)
            s = list(rng.permutation(s))
        d = np.cumsum([1 if c == "(" else -1 for c in s])
        xs.append([VOCAB[c] for c in s])
        ys.append(int(d[-1] == 0 and d.min() >= 0))
        depths.append(d)
    return torch.tensor(xs), torch.tensor(ys), np.array(depths)

---
### 🕐 Session 1 of 2 — *Probes & Attention Maps* (~40 min)
**Goal:** train the model; find WHERE the running depth lives with linear probes.
**Builds on:** [Transformers workshop](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb). &nbsp; **Feeds into:** Session 2 (activation patching).

---

## 2. The Model, and the Hypothesis

💡 **Intuition.** Balanced-parentheses has a one-line algorithm: track the running depth, check it never dips below zero and ends at zero. If our transformer learns the task, *something inside it should represent the running depth*. A **linear probe** — a tiny regression from hidden states to the known quantity — tests exactly that, layer by layer and position by position. Finding a probe that works is evidence of a representation; Session 2 tests whether the model actually *uses* it.

In [ ]:

# YOUR CODE HERE


In [ ]:
# probe every layer for the RUNNING DEPTH (per position) — where does the algorithm live?

# YOUR CODE HERE


---
### 🕐 Session 2 of 2 — *Activation Patching: the Causal Test* (~40 min)
**Goal:** swap internal activations between clean and corrupted runs — which components MATTER?
**Builds on:** Session 1; [Causal Inference](./Causal_Inference.ipynb).

---

## 3. From Correlation to Mechanism

💡 **Intuition.** A probe finding depth proves the information is *present*, not that it's *used* — the [collider lesson](./Causal_Inference.ipynb) for neural nets. **Activation patching** is the intervention: run a balanced string and an unbalanced one; copy one layer's activations from the balanced run into the unbalanced run; if the verdict flips toward 'balanced', that layer *causally carries* the verdict. Do it per layer and position and you map the circuit. This do-operator-for-networks is the core method of modern interpretability research.

In [ ]:
# clean = balanced string; corrupted = same string with ONE paren flipped
# single-position patches are diluted by the 16-position mean-pool — patch WHOLE layers too

# YOUR CODE HERE


## 4. Conclusion

Probes locate representations (depth R² rising through the blocks); patching tests *use* (verdict recovery mapped by layer and position). Correlation-to-causation, inside the network — the same discipline [Causal Inference](./Causal_Inference.ipynb) taught for the world outside. Scaling these methods to frontier models is an open, hiring-hot research field.

---
## Where next

- [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb) — apply both tools to the fable nano-GPT you trained there.
- [Causal Inference](./Causal_Inference.ipynb) — the intervention logic, formalized.